In [6]:
import pandas as pd
import numpy as np
import pyodbc
import warnings

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)

def run_sql(query):
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=conn)
        warnings.filterwarnings("default", category=UserWarning)
    return df

with pyodbc.connect("DSN=Redshift_prod_new") as conn:
    conn.cursor().execute("SELECT 1").fetchone()
print("ODBC connection OK")

ODBC connection OK


In [7]:
query_a = """
SELECT
    ldcf.loan_id,
    ldcf.account_number,
    ldcf.aspect,
    dru.riskdealergroup AS lob,
    ldcf.dealer_pricing_hurdle,
    ldcf.data_source_id,
    ldcf.con_amount_financed_back,
    ldcf.con_pti_back,
    COALESCE(ldcf.pb_monthly_income_gross_total, 0)
        + COALESCE(ldcf.cb_monthly_income_gross_total, 0) AS total_income,
    ldcf.blackbook_history_adj_wholesale_amt AS bb_value,
    CASE WHEN ldcf.blackbook_history_adj_wholesale_amt IS NOT NULL
              AND ldcf.blackbook_history_adj_wholesale_amt > 0
         THEN ldcf.con_amount_financed_back / ldcf.blackbook_history_adj_wholesale_amt
         ELSE NULL END AS bbltv,

    CASE
        WHEN ldcf.dealer_pricing_hurdle = 'mROA-FLD'
            THEN CASE WHEN COALESCE(lrn.maxltv, 0) > 750 THEN '1) Lower' ELSE '2) Higher' END
        WHEN ldcf.data_source_id = 101
            THEN CASE WHEN COALESCE(lrn.maxltv, 0) < 500 THEN '2) Higher' ELSE '1) Lower' END
        WHEN COALESCE(lrn.maxltv, 0) < 500
            OR (COALESCE(lrn.maxltv, 0) < 850
                AND ldcf.dealer_pricing_hurdle = 'mROA-MCY'
                AND ldcf.application_received_dtm >= '2024-06-19')
            THEN '2) Higher'
        ELSE '1) Lower'
    END AS hurdle,

    lrn.loan_id AS lrn_loan_id,
    ragu.account_number AS ragu_account_number

FROM los_deal_current_fact ldcf

INNER JOIN edwnpi.dealer_rollup_scd_current dru
    ON dru.dealer_number = ldcf.dealer_number

LEFT JOIN sandbox.loan_random_numbers lrn
    ON lrn.loan_id = ldcf.loan_id

LEFT JOIN sandbox.gl_ragu_individual ragu
    ON ragu.account_number = ldcf.account_number

WHERE ldcf.application_received_dtm >= '2026-05-01'
  AND ldcf.application_received_dtm < '2026-06-01'
  AND aspect = 'CONTRACT'
  AND dru.riskdealergroup IN ('AN', 'ENT', 'FLD', 'FRN', 'STG', 'MCY')
"""

query_b = """
SELECT
    ldcf.loan_id,
    ldcf.account_number,
    ldcf.aspect,
    dru.riskdealergroup AS lob,
    ldcf.dealer_pricing_hurdle,
    ldcf.con_amount_financed_back,
    ldcf.con_pti_back,
    COALESCE(ldcf.pb_monthly_income_gross_total, 0)
        + COALESCE(ldcf.cb_monthly_income_gross_total, 0) AS total_income,
    ldcf.blackbook_history_adj_wholesale_amt AS bb_value,
    CASE WHEN ldcf.blackbook_history_adj_wholesale_amt IS NOT NULL
              AND ldcf.blackbook_history_adj_wholesale_amt > 0
         THEN ldcf.con_amount_financed_back / ldcf.blackbook_history_adj_wholesale_amt
         ELSE NULL END AS bbltv

FROM los_deal_current_fact ldcf

INNER JOIN edwnpi.dealer_rollup_scd_current dru
    ON dru.dealer_number = ldcf.dealer_number

WHERE ldcf.application_received_dtm >= '2026-05-01'
  AND ldcf.application_received_dtm < '2026-06-01'
  AND (ldcf.book_date >= '2020-01-01' OR (ldcf.aspect = 'CONTRACT' AND ldcf.book_date IS NULL))
  AND ldcf.data_source_name != 'SPARTAN'
  AND dru.riskdealergroup IN ('AN', 'ENT', 'FLD', 'FRN', 'STG', 'MCY')
  AND ldcf.account_number != 90124841967
"""

print("Running Query A (ragu method with joins)...")
df_a = run_sql(query_a)
print(f"  Query A: {len(df_a)} rows, {df_a['account_number'].nunique()} distinct accounts")

print("Running Query B (weekly method, no LRN/RAGU joins)...")
df_b = run_sql(query_b)
print(f"  Query B: {len(df_b)} rows, {df_b['account_number'].nunique()} distinct accounts")

Running Query A (ragu method with joins)...
  Query A: 7295 rows, 5462 distinct accounts
Running Query B (weekly method, no LRN/RAGU joins)...
  Query B: 5867 rows, 5462 distinct accounts


In [8]:
def apply_weekly_filters(df):
    """Apply the credit/amount filters from weekly copy 2.ipynb."""
    out = df.copy()
    out = out[out['con_amount_financed_back'] <= 75000]
    out = out[out['con_pti_back'] <= 0.6]
    out = out[out['total_income'] <= 200000]
    out = out[(out['lob'] == 'MCY') | (out['bbltv'] <= 10.0) | (out['bb_value'].isna()) | (out['bb_value'] == 0)]
    return out

print(f"Query B aspects: {df_b['aspect'].value_counts().to_dict()}")
print(f"Query B total rows: {len(df_b)}, distinct accounts: {df_b['account_number'].nunique()}")

lobs = sorted(df_a['lob'].unique())
rows = []

for lob in lobs:
    a_lob = df_a[df_a['lob'] == lob]
    b_lob = df_b[df_b['lob'] == lob]

    count_star = len(a_lob)
    distinct_accts = a_lob['account_number'].nunique()

    a_filtered = apply_weekly_filters(a_lob)
    after_filters = a_filtered['account_number'].nunique()

    weekly_distinct = b_lob['account_number'].nunique()
    b_filtered = apply_weekly_filters(b_lob)
    weekly_after_filters = b_filtered['account_number'].nunique()

    rows.append({
        'lob': lob,
        'A_count_star': count_star,
        'A_distinct_accts': distinct_accts,
        'A_dup_rows': count_star - distinct_accts,
        'A_after_credit_filters': after_filters,
        'A_excluded_by_filters': distinct_accts - after_filters,
        'B_distinct_accts': weekly_distinct,
        'B_after_credit_filters': weekly_after_filters,
        'gap_A_star_vs_B_filtered': count_star - weekly_after_filters,
    })

summary = pd.DataFrame(rows)

totals = summary.select_dtypes(include='number').sum()
totals['lob'] = 'TOTAL'
summary = pd.concat([summary, pd.DataFrame([totals])], ignore_index=True)

print()
print("=" * 110)
print("  LOAN COUNT COMPARISON -- May 2026, Non-KMX")
print("  A = ragu method (aspect=CONTRACT + LRN + RAGU joins)")
print("  B = weekly method (book_date filter, no aspect filter, excl SPARTAN)")
print("=" * 110)
print()
print(summary.to_string(index=False))
print()
print("-" * 110)
print("  A_count_star:           COUNT(*) from ragu query (what ragu_by_hurdle_pool.sql reports)")
print("  A_distinct_accts:       COUNT(DISTINCT account_number) from same data")
print("  A_dup_rows:             extra rows from join fan-out or multiple fact rows")
print("  A_after_credit_filters: distinct accounts after weekly-style credit filters")
print("  B_distinct_accts:       distinct accounts from weekly query (no aspect filter)")
print("  B_after_credit_filters: distinct accounts from weekly query + credit filters")
print("  gap:                    A_count_star - B_after_credit_filters (total discrepancy)")
print("-" * 110)

Query B aspects: {'CONTRACT': 5867}
Query B total rows: 5867, distinct accounts: 5462

  LOAN COUNT COMPARISON -- May 2026, Non-KMX
  A = ragu method (aspect=CONTRACT + LRN + RAGU joins)
  B = weekly method (book_date filter, no aspect filter, excl SPARTAN)

  lob  A_count_star  A_distinct_accts  A_dup_rows  A_after_credit_filters  A_excluded_by_filters  B_distinct_accts  B_after_credit_filters  gap_A_star_vs_B_filtered
   AN          1053               779         274                     776                      3               779                     776                       277
  ENT           741               579         162                     578                      1               579                     578                       163
  FLD           873               688         185                     685                      3               688                     685                       188
  FRN          2750              2066         684                    2058        

In [9]:
# --- PART 1: Duplicate account_numbers in Query A ---
dup_counts = df_a.groupby('account_number').size()
dups = dup_counts[dup_counts > 1]

print("=" * 90)
print(f"  DUPLICATE ACCOUNT NUMBERS (Query A) -- {len(dups)} accounts with multiple rows")
print("=" * 90)

if len(dups) > 0:
    print(f"\n  Distribution of row counts per duplicate account:")
    print(f"  {dups.value_counts().sort_index().to_dict()}")

    dup_detail = df_a[df_a['account_number'].isin(dups.index)].sort_values(['account_number', 'loan_id'])
    print(f"\n  Sample duplicate accounts (first 20):")
    sample_accts = dups.head(20).index
    sample = dup_detail[dup_detail['account_number'].isin(sample_accts)]
    cols_to_show = ['account_number', 'loan_id', 'lob', 'hurdle', 'lrn_loan_id', 'ragu_account_number']
    print(sample[cols_to_show].to_string(index=False))

    lrn_fanout = dup_detail.groupby('account_number')['lrn_loan_id'].nunique()
    ragu_fanout = dup_detail.groupby('account_number')['ragu_account_number'].nunique()
    loanid_fanout = dup_detail.groupby('account_number')['loan_id'].nunique()

    print(f"\n  Among {len(dups)} duplicate accounts:")
    print(f"    Multiple loan_ids:          {(loanid_fanout > 1).sum()}")
    print(f"    Multiple lrn matches:       {(lrn_fanout > 1).sum()}")
    print(f"    Multiple ragu matches:      {(ragu_fanout > 1).sum()}")
else:
    print("\n  No duplicate account_numbers found.")

# --- PART 2: Contracts excluded by each credit filter ---
print("\n" + "=" * 90)
print("  CONTRACTS EXCLUDED BY CREDIT FILTERS (Query A, distinct accounts)")
print("=" * 90)

total_distinct = df_a.drop_duplicates('account_number')

filters = {
    'con_amount_financed_back > 75000': total_distinct['con_amount_financed_back'] > 75000,
    'con_pti_back > 0.6': total_distinct['con_pti_back'] > 0.6,
    'total_income > 200000': total_distinct['total_income'] > 200000,
    'bbltv > 10 (non-MCY)': (
        (total_distinct['lob'] != 'MCY') &
        (total_distinct['bbltv'] > 10.0) &
        (total_distinct['bb_value'].notna()) &
        (total_distinct['bb_value'] != 0)
    ),
}

print(f"\n  Total distinct accounts: {len(total_distinct)}")
print()
for name, mask in filters.items():
    excluded = mask.sum()
    by_lob = total_distinct.loc[mask, 'lob'].value_counts().sort_index().to_dict()
    print(f"  {name}: {excluded} excluded")
    if excluded > 0:
        print(f"    By LOB: {by_lob}")

any_filter = pd.Series(False, index=total_distinct.index)
for mask in filters.values():
    any_filter = any_filter | mask
print(f"\n  Excluded by ANY filter:  {any_filter.sum()}")
print(f"  Remaining after filters: {(~any_filter).sum()}")

  DUPLICATE ACCOUNT NUMBERS (Query A) -- 350 accounts with multiple rows

  Distribution of row counts per duplicate account:
  {2: 286, 3: 50, 4: 12, 5: 1, 6: 1}

  Sample duplicate accounts (first 20):
 account_number  loan_id lob    hurdle  lrn_loan_id  ragu_account_number
   9.012526e+10 40020555 FLD 2) Higher     40020555         9.012526e+10
   9.012526e+10 40020555 FLD 2) Higher     40020555         9.012526e+10
   9.012526e+10 39998700 FRN 2) Higher     39998700         9.012526e+10
   9.012526e+10 39998700 FRN 2) Higher     39998700         9.012526e+10
   9.012526e+10 40025439 FRN  1) Lower     40025439         9.012526e+10
   9.012526e+10 40025439 FRN  1) Lower     40025439         9.012526e+10
   9.012526e+10 40054485 FRN 2) Higher     40054485         9.012526e+10
   9.012526e+10 40054485 FRN 2) Higher     40054485         9.012526e+10
   9.012526e+10 40054868 FRN 2) Higher     40054868         9.012526e+10
   9.012526e+10 40054868 FRN 2) Higher     40054868         9.0125